### Process Orders Data

##### 1. Ingest the data into the data lakehouse - bronze_orders
##### 2. Perform data quality checks and transform the data as required - silver_orders_clean
##### 3. Explode the items array from the order object - silver_orders

![](/Volumes/circuitbox/landing/operational_data/images_dlt/DLT ST_MV_Views.png)

![](/Volumes/circuitbox/landing/operational_data/images_dlt/Gold_Customer_Order_Summary.png)

In [0]:
CREATE OR REFRESH STREAMING TABLE bronze_orders
COMMENT "Raw orders data ingested from the Source System"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT *,
      _metadata.file_path as input_file_path,
      CURRENT_TIMESTAMP as ingested_timestamp
FROM cloud_files(
        '/Volumes/circuitbox/landing/operational_data/orders/',
        'json',
        map("cloudFiles.inferColumnTypes", "true")
      );

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_orders_clean(
CONSTRAINT valid_customers_id EXPECT (customer_id IS NOT NULL) ON 
VIOLATION FAIL UPDATE,
CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
CONSTRAINT valid_order_status EXPECT (order_status IN ("Pending", "Shipped", "Cancelled", "Completed")),
CONSTRAINT valid_payment_method EXPECT (payment_method IN ("Credit Card", "Bank Transfer", "PayPal"))
)
COMMENT "Cleaned Orders Data"
TBLPROPERTIES ("quality" = "silver_cleaned")
AS
SELECT order_id,
      customer_id,
      CAST(order_timestamp AS TIMESTAMP) as order_timestamp,
      payment_method,
      items,
      order_status
FROM STREAM(LIVE.bronze_orders);

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_orders
AS
SELECT order_id,
      customer_id,
      order_timestamp,
      payment_method,
      items,
      order_status,
      item.item_id,
      item.name as item_name,
      item.price as item_price,
      item.quantity as item_quantity,
      item.category as item_category
FROM (
    SELECT order_id,
      customer_id,
      order_timestamp,
      payment_method,
      items,
      order_status,
      explode(items) as item
    FROM STREAM(LIVE.silver_orders_clean
    )
);